# NLI classifier + cleaned synthetic misrepresentations (v2)

The first synthetic run (`results/synthetic_v1.md`) raised accuracy on every real misrepresentation
label by 13 to 39 points, but lost 11 on accurate citations, because the LLM's rewrites were tidier
than real citing sentences (a style shortcut) and some were not real edits. `scripts/clean_misrep.py`
fixed both: 975 cleaned rewrites are in `data_train/synthetic/misrep_clean.jsonl`, so nothing is
generated here.

- **Phase 1** (about 2.5 hours): 3-fold cross-validation with all the cleaned synthetic data, then
  with half of it. The goal is to beat the NLI model without synthetic data (CV macro-F1 0.570).
- **Phase 2** (about 2 hours): 3 seeds of the better variant, predicting test. **It runs only if that
  variant beats 0.570.**

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access. Everything is saved to `MyDrive/CiteAudit_runs/v7`; if Colab disconnects, Run
all again and finished runs are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v7'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS',
             'Loading weights', 'Warning', 'warn(')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
SYN = 'data_train/synthetic/misrep_clean.jsonl'
def train(name, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_nli.py --train {T}/ours_20r_train_cv --dev {T}/ours_20r '
        f'--epochs 3 --batch 2 --accum 4 --seed {seed} --amp --synthetic {SYN} '
        f'--out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]
COLS = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']

## Phase 1: 3-fold cross-validation, all and half of the cleaned synthetic data

In [ ]:
VARIANTS = {'full': 1.0, 'half': 0.5}
BASE = [0.676, 0.570, 0.783, 0.513, 0.415]   # NLI without synthetic data, same folds (nli_classifier_v1.md)
cv = {}
for v, frac in VARIANTS.items():
    for k in range(3):
        train(f'cv_{v}_f{k}', 0, f'--folds 3 --fold {k} --synth-frac {frac}')
    cv[v] = scores(sum((read(f'{RES}/cv_{v}_f{k}/oof.jsonl') for k in range(3)), []))
print(f'{"cross-validated, 2141 citations":32s} ' + ' '.join(f'{c:>9s}' for c in COLS))
print(f'{"NLI, real data only":32s} ' + ' '.join(f'{x:9.3f}' for x in BASE))
print(f'{"NLI, raw synthetic (v1)":32s} ' + ' '.join(f'{x:9.3f}' for x in [0.629, 0.542, 0.724, 0.518, 0.386]))
for v in VARIANTS:
    print(f'{"NLI, cleaned synthetic, " + v:32s} ' + ' '.join(f'{x:9.3f}' for x in cv[v]))
BEST = max(VARIANTS, key=lambda v: cv[v][1])
GO = cv[BEST][1] > BASE[1]
json.dump({'cv': cv, 'nli_cv': BASE, 'best': BEST, 'phase2': GO}, open(f'{RES}/cv_summary.json', 'w'), indent=2)
print(f'\nPhase 2 will run with "{BEST}".' if GO else '\nNeither variant beats real data only; Phase 2 is skipped.')

## Phase 2: 3 seeds of the better variant on all of train, predicting test (only if Phase 1 won)

In [ ]:
if GO:
    for seed in (0, 1, 2):
        train(f'final_{BEST}_s{seed}', seed, f'--synth-frac {VARIANTS[BEST]} --predict test={T}/ours_20r_test')
    runs = [scores(read(f'{RES}/final_{BEST}_s{s}/ours_20r_test.jsonl')) for s in (0, 1, 2)]
    print('test, mean ± sd over 3 seeds')
    for name, xs in zip(COLS, zip(*runs)):
        print(f'  {name:9s} {statistics.mean(xs):.3f} ± {statistics.stdev(xs):.3f}')
    print('NLI without synthetic data on test: accuracy 0.668 ± 0.015, macro-F1 0.556 ± 0.015')

## Download

Zips `MyDrive/CiteAudit_runs/v7` (predictions and logs, no weights). Put `v7.zip` in the repository
root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v7.zip v7 -x '*.part/*'
from google.colab import files
files.download('/content/v7.zip')